# PathSnap — ML Model Experimentation Framework

This notebook builds and tests the candidate-level ML framework before Shayna's real candidate features and labels arrive.

**Important:** All data in this notebook is synthetic and used only to test the pipeline and integration interface. Do not report synthetic scores as PathSnap results.

Workflow:
1. Environment setup
2. Synthetic candidate dataset
3. Feature and target preparation
4. Five initial model configurations
5. Evaluation functions
6. Five-fold GroupKFold cross-validation
7. Candidate and observation-level metrics
8. Probability normalisation
9. `predict_candidates` interface for the HMM decoder
10. Validation tests

**Tuning rule:** This notebook does not tune hyperparameters. After real-data baseline experiments, use Bayesian optimisation only. Do not use Grid Search or Random Search.

## 1. Environment setup

In [1]:
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

from sklearn.base import clone
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, log_loss, confusion_matrix, classification_report
)

RANDOM_STATE = 42
N_SPLITS = 5
np.random.seed(RANDOM_STATE)

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)

Python: 3.13.18
NumPy: 2.4.6
Pandas: 2.2.3
Scikit-learn: 1.6.1


## 2. Synthetic candidate-level dataset

Relevant candidates are given illustrative feature patterns (for example, lower distance and heading difference). This makes the data useful for testing whether the code runs and learns a signal.

These are **not actual PathSnap feature distributions**. Replace this dataset with Shayna's real candidate features and labels for real experiments.

In [2]:
def create_synthetic_candidate_data(
    n_trips=10,
    observations_per_trip=20,
    candidates_per_observation=4,
    random_state=42
):
    rng = np.random.default_rng(random_state)
    records = []

    for trip_number in range(n_trips):
        trip_id = f"synthetic_trip_{trip_number:03d}"

        for obs_idx in range(observations_per_trip):
            positive_candidate = rng.integers(0, candidates_per_observation)

            for candidate_number in range(candidates_per_observation):
                is_correct = int(candidate_number == positive_candidate)

                if is_correct:
                    distance = rng.gamma(shape=2.0, scale=5.0)
                    heading_difference = abs(rng.normal(8, 8))
                    connectivity_score = rng.uniform(0.65, 1.0)
                else:
                    distance = rng.uniform(15, 100)
                    heading_difference = rng.uniform(15, 180)
                    connectivity_score = rng.uniform(0.0, 0.75)

                road_length = rng.uniform(10, 500)
                candidate_arc_id = (
                    trip_number * 10000
                    + obs_idx * candidates_per_observation
                    + candidate_number
                )

                records.append({
                    "trip_id": trip_id,
                    "obs_idx": obs_idx,
                    "candidate_arc_id": candidate_arc_id,
                    "distance": distance,
                    "heading_difference": min(heading_difference, 180),
                    "road_length": road_length,
                    "connectivity_score": connectivity_score,
                    "label": is_correct
                })

    df = pd.DataFrame(records)
    return df.sample(frac=1, random_state=random_state).reset_index(drop=True)


synthetic_df = create_synthetic_candidate_data()
print("Dataset shape:", synthetic_df.shape)
print("Unique trips:", synthetic_df["trip_id"].nunique())
display(synthetic_df.head(10))
display(synthetic_df["label"].value_counts())

Dataset shape: (800, 8)
Unique trips: 10


,trip_id,obs_idx,candidate_arc_id,distance,heading_difference,road_length,connectivity_score,label
0,synthetic_trip_008,14,80056,52.912696,82.600678,368.497238,0.136457,0
1,synthetic_trip_008,6,80027,35.793306,16.845814,119.016304,0.739969,0
2,synthetic_trip_000,15,63,18.695380,44.947238,132.199912,0.177559,0
3,synthetic_trip_006,13,60053,3.807372,4.522404,399.807508,0.799185,1
4,synthetic_trip_000,16,66,1.383057,11.635363,402.992681,0.992850,1
5,synthetic_trip_007,15,70061,30.429464,102.388165,291.894527,0.530022,0
6,synthetic_trip_004,6,40026,61.770374,131.329652,245.750320,0.052449,0
7,synthetic_trip_006,2,60010,9.681999,5.926326,142.467076,0.957794,1
8,synthetic_trip_009,10,90040,82.619642,112.288101,202.427929,0.580387,0
9,synthetic_trip_005,14,50056,52.491582,83.368372,147.021801,0.391407,0


,count
label,
0,600
1,200


## 3. Feature and target preparation
Identifiers are kept for grouping and output, not used as model features.

In [3]:
IDENTIFIER_COLUMNS = ["trip_id", "obs_idx", "candidate_arc_id"]
TARGET_COLUMN = "label"

FEATURE_COLUMNS = [
    "distance",
    "heading_difference",
    "road_length",
    "connectivity_score"
]

X = synthetic_df[FEATURE_COLUMNS].copy()
y = synthetic_df[TARGET_COLUMN].astype(int).copy()
groups = synthetic_df["trip_id"].copy()

assert not X.isnull().any().any()
assert set(y.unique()).issubset({0, 1})
assert len(X) == len(y) == len(groups)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Unique trips:", groups.nunique())
display(X.describe())

X shape: (800, 4)
y shape: (800,)
Unique trips: 10


,distance,heading_difference,road_length,connectivity_score
count,800.000000,800.000000,800.000000,800.000000
mean,46.624288,73.533582,250.816828,0.491105
std,30.345165,56.468038,140.536317,0.275765
min,0.294589,0.019006,10.278383,0.000389
25%,17.956939,17.832752,129.246775,0.245547
50%,46.639370,64.387605,246.550170,0.527291
75%,72.503954,124.412267,368.680676,0.708381
max,99.948153,179.961281,498.814218,0.998606


## 4. Five initial model configurations
These are baseline configurations, not tuned hyperparameters.

In [4]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    ]),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=6, min_samples_split=4, random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100, max_depth=8, min_samples_split=4,
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100, learning_rate=0.1, max_depth=3,
        random_state=RANDOM_STATE
    ),
    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=5))
    ])
}

for name, model in models.items():
    print(name, ":", model)
print("Total models:", len(models))

Logistic Regression : Pipeline(steps=[('scaler', StandardScaler()),
                ('model', LogisticRegression(max_iter=1000, random_state=42))])
Decision Tree : DecisionTreeClassifier(max_depth=6, min_samples_split=4, random_state=42)
Random Forest : RandomForestClassifier(max_depth=8, min_samples_split=4, n_jobs=-1,
                       random_state=42)
Gradient Boosting : GradientBoostingClassifier(random_state=42)
KNN : Pipeline(steps=[('scaler', StandardScaler()),
                ('model', KNeighborsClassifier())])
Total models: 5


## 5. Common evaluation functions

In [5]:
def get_positive_class_probability(model, X_data):
    if not hasattr(model, "predict_proba"):
        raise TypeError("Model must support predict_proba().")

    classes = np.asarray(model.classes_)
    positive_indices = np.where(classes == 1)[0]

    if len(positive_indices) != 1:
        raise ValueError("Fitted model does not contain positive class 1.")

    return model.predict_proba(X_data)[:, positive_indices[0]]


def calculate_classification_metrics(y_true, y_pred, positive_probability):
    positive_probability = np.asarray(positive_probability, dtype=float)
    positive_probability = np.clip(positive_probability, 1e-15, 1 - 1e-15)

    probability_matrix = np.column_stack([
        1 - positive_probability,
        positive_probability
    ])

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "Macro_F1": f1_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "Log_Loss": log_loss(y_true, probability_matrix, labels=[0, 1])
    }


def calculate_top1_accuracy(prediction_df, true_labels):
    evaluation_df = prediction_df[
        ["trip_id", "obs_idx", "probability"]
    ].copy()
    evaluation_df["true_label"] = np.asarray(true_labels)

    selected_indices = (
        evaluation_df.groupby(
            ["trip_id", "obs_idx"], sort=False
        )["probability"].idxmax()
    )
    selected = evaluation_df.loc[selected_indices]

    return accuracy_score(
        np.ones(len(selected), dtype=int),
        selected["true_label"].to_numpy()
    )

print("Evaluation functions defined.")

Evaluation functions defined.


## 6. Five-model GroupKFold cross-validation

Splitting is by trip, not by individual candidate rows. Out-of-fold predictions are collected so each row is evaluated by a model that did not train on that trip.

In [6]:
if groups.nunique() < N_SPLITS:
    raise ValueError(f"Need at least {N_SPLITS} unique trips.")

group_kfold = GroupKFold(n_splits=N_SPLITS)
cv_results = []
oof_predictions = {}

for model_name, base_model in models.items():
    print("\n" + "=" * 60)
    print("MODEL:", model_name)
    print("=" * 60)

    oof_probability = np.full(len(X), np.nan, dtype=float)
    fold_training_times = []
    fold_inference_times = []

    for fold, (train_idx, validation_idx) in enumerate(
        group_kfold.split(X, y, groups=groups), start=1
    ):
        X_train, X_validation = X.iloc[train_idx], X.iloc[validation_idx]
        y_train = y.iloc[train_idx]

        train_groups = set(groups.iloc[train_idx])
        validation_groups = set(groups.iloc[validation_idx])
        assert train_groups.isdisjoint(validation_groups), "Trip leakage detected."

        model = clone(base_model)

        start = time.perf_counter()
        model.fit(X_train, y_train)
        training_time = time.perf_counter() - start

        start = time.perf_counter()
        validation_probability = get_positive_class_probability(
            model, X_validation
        )
        inference_time = time.perf_counter() - start

        oof_probability[validation_idx] = validation_probability
        fold_training_times.append(training_time)
        fold_inference_times.append(inference_time)

        print(
            f"Fold {fold}: train trips={len(train_groups)}, "
            f"validation trips={len(validation_groups)}, "
            f"train={training_time:.4f}s, inference={inference_time:.4f}s"
        )

    assert np.isfinite(oof_probability).all()

    oof_predicted_label = (oof_probability >= 0.5).astype(int)
    metrics = calculate_classification_metrics(
        y, oof_predicted_label, oof_probability
    )

    prediction_df = synthetic_df[
        ["trip_id", "obs_idx", "candidate_arc_id"]
    ].copy()
    prediction_df["probability"] = oof_probability

    top1_accuracy = calculate_top1_accuracy(prediction_df, y)

    result = {
        "Model": model_name,
        **metrics,
        "Top1_Accuracy": top1_accuracy,
        "Avg_Training_Time_Seconds": float(np.mean(fold_training_times)),
        "Avg_Inference_Time_Seconds": float(np.mean(fold_inference_times))
    }

    cv_results.append(result)
    oof_predictions[model_name] = prediction_df

cv_results_df = pd.DataFrame(cv_results)
display(cv_results_df.sort_values("Macro_F1", ascending=False).reset_index(drop=True))


MODEL: Logistic Regression
Fold 1: train trips=8, validation trips=2, train=0.0612s, inference=0.0045s
Fold 2: train trips=8, validation trips=2, train=0.0303s, inference=0.0033s
Fold 3: train trips=8, validation trips=2, train=0.0626s, inference=0.0166s
Fold 4: train trips=8, validation trips=2, train=0.0638s, inference=0.0029s
Fold 5: train trips=8, validation trips=2, train=0.0232s, inference=0.0027s

MODEL: Decision Tree
Fold 1: train trips=8, validation trips=2, train=0.0160s, inference=0.0052s
Fold 2: train trips=8, validation trips=2, train=0.0107s, inference=0.0065s
Fold 3: train trips=8, validation trips=2, train=0.0101s, inference=0.0040s
Fold 4: train trips=8, validation trips=2, train=0.0115s, inference=0.0045s
Fold 5: train trips=8, validation trips=2, train=0.0123s, inference=0.0057s

MODEL: Random Forest
Fold 1: train trips=8, validation trips=2, train=0.4312s, inference=0.0441s
Fold 2: train trips=8, validation trips=2, train=0.5187s, inference=0.0259s
Fold 3: train tr

,Model,Accuracy,Precision,Recall,Macro_F1,Log_Loss,Top1_Accuracy,Avg_Training_Time_Seconds,Avg_Inference_Time_Seconds
0,Random Forest,0.99625,0.994975,0.990,0.994992,0.055867,0.995,0.475734,0.046477
1,Logistic Regression,0.99375,0.980296,0.995,0.991708,0.024279,1.000,0.048216,0.006001
2,KNN,0.99250,0.975490,0.995,0.990066,0.140394,0.995,0.018893,0.017472
3,Gradient Boosting,0.99125,0.984925,0.980,0.988314,0.032796,0.995,0.504414,0.003839
4,Decision Tree,0.98500,0.965347,0.975,0.980066,0.518595,0.965,0.012140,0.005150


## 7. Probability normalisation per GPS observation

Binary positive-class scores are produced independently for candidates and do not necessarily sum to one within an observation. This function normalises scores within each `(trip_id, obs_idx)` group. All-zero groups receive a uniform distribution.

In [7]:
def normalize_candidate_probabilities(candidate_df, score_column="probability"):
    required_columns = ["trip_id", "obs_idx", score_column]
    missing = [c for c in required_columns if c not in candidate_df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    result = candidate_df.copy()

    if result[["trip_id", "obs_idx"]].isnull().any().any():
        raise ValueError("trip_id and obs_idx cannot be missing.")

    scores = pd.to_numeric(
        result[score_column], errors="coerce"
    ).to_numpy(dtype=float)

    if not np.isfinite(scores).all():
        raise ValueError("Scores must be finite numeric values.")
    if (scores < 0).any():
        raise ValueError("Scores cannot be negative.")

    result[score_column] = scores
    keys = ["trip_id", "obs_idx"]

    sums = result.groupby(keys, sort=False)[score_column].transform("sum")
    sizes = result.groupby(keys, sort=False)[score_column].transform("size")

    positive = sums > 0
    result.loc[positive, score_column] = (
        result.loc[positive, score_column] / sums.loc[positive]
    )

    zero = ~positive
    result.loc[zero, score_column] = 1.0 / sizes.loc[zero]

    return result


print("Normalisation function defined.")

Normalisation function defined.


## 8. `predict_candidates` interface for the HMM decoder

Expected output columns:
- `trip_id`
- `obs_idx`
- `candidate_arc_id`
- `probability`

Every input candidate row is retained. Probabilities are normalised within each observation group.

In [8]:
def predict_candidates(candidate_df, trained_model, feature_columns):
    identifier_columns = ["trip_id", "obs_idx", "candidate_arc_id"]
    required = identifier_columns + list(feature_columns)

    missing = [c for c in required if c not in candidate_df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")
    if candidate_df.empty:
        raise ValueError("candidate_df cannot be empty.")
    if candidate_df[identifier_columns].isnull().any().any():
        raise ValueError("Candidate identifiers cannot be missing.")
    if not hasattr(trained_model, "predict_proba"):
        raise TypeError("Model must support predict_proba().")

    classes = np.asarray(trained_model.classes_)
    positive_indices = np.where(classes == 1)[0]
    if len(positive_indices) != 1:
        raise ValueError("Fitted model must contain class label 1.")

    scores = trained_model.predict_proba(
        candidate_df[feature_columns]
    )[:, positive_indices[0]]

    output = candidate_df[identifier_columns].copy()
    output["probability"] = scores
    output = normalize_candidate_probabilities(output)

    return output[
        ["trip_id", "obs_idx", "candidate_arc_id", "probability"]
    ]


print("Prediction interface defined.")

Prediction interface defined.


## 9. Test the prediction interface
This is a smoke test on synthetic data, not an unbiased performance estimate.

In [9]:
interface_test_model = clone(models["Random Forest"])
interface_test_model.fit(X, y)

interface_predictions = predict_candidates(
    candidate_df=synthetic_df,
    trained_model=interface_test_model,
    feature_columns=FEATURE_COLUMNS
)

expected_columns = [
    "trip_id", "obs_idx", "candidate_arc_id", "probability"
]

display(interface_predictions.head(12))

assert list(interface_predictions.columns) == expected_columns
assert len(interface_predictions) == len(synthetic_df)
assert interface_predictions["probability"].notna().all()
assert np.isfinite(interface_predictions["probability"]).all()
assert (interface_predictions["probability"] >= 0).all()
assert (interface_predictions["probability"] <= 1).all()

probability_sums = interface_predictions.groupby(
    ["trip_id", "obs_idx"]
)["probability"].sum()

assert np.allclose(probability_sums.to_numpy(), 1.0, atol=1e-8)

print("Prediction interface checks passed.")
print("Candidate predictions:", len(interface_predictions))
print("GPS observations:", len(probability_sums))

,trip_id,obs_idx,candidate_arc_id,probability
0,synthetic_trip_008,14,80056,0.000000
1,synthetic_trip_008,6,80027,0.180104
2,synthetic_trip_000,15,63,0.000000
3,synthetic_trip_006,13,60053,0.980232
4,synthetic_trip_000,16,66,1.000000
5,synthetic_trip_007,15,70061,0.000000
6,synthetic_trip_004,6,40026,0.000000
7,synthetic_trip_006,2,60010,1.000000
8,synthetic_trip_009,10,90040,0.000000
9,synthetic_trip_005,14,50056,0.000000


Prediction interface checks passed.
Candidate predictions: 800
GPS observations: 200


## 10. Test all-zero score handling

In [10]:
zero_score_test = pd.DataFrame({
    "trip_id": ["trip_01", "trip_01", "trip_01"],
    "obs_idx": [0, 0, 0],
    "candidate_arc_id": [101, 102, 103],
    "probability": [0.0, 0.0, 0.0]
})

zero_score_result = normalize_candidate_probabilities(zero_score_test)
display(zero_score_result)

assert np.allclose(
    zero_score_result["probability"].to_numpy(),
    [1/3, 1/3, 1/3]
)
print("All-zero score test passed.")

,trip_id,obs_idx,candidate_arc_id,probability
0,trip_01,0,101,0.333333
1,trip_01,0,102,0.333333
2,trip_01,0,103,0.333333


All-zero score test passed.


## 11. Final framework validation

In [11]:
checks = {
    "Synthetic data created": not synthetic_df.empty,
    "Five models defined": len(models) == 5,
    "Five-model CV completed": len(cv_results_df) == 5,
    "Out-of-fold predictions available": len(oof_predictions) == 5,
    "Prediction output schema correct": (
        list(interface_predictions.columns) == expected_columns
    ),
    "All candidate rows retained": (
        len(interface_predictions) == len(synthetic_df)
    ),
    "Probabilities sum to one per observation": (
        np.allclose(probability_sums.to_numpy(), 1.0, atol=1e-8)
    )
}

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")

assert all(checks.values()), "One or more framework checks failed."
print("\nAll framework validation checks passed.")
print("Reminder: synthetic framework tests only, not real PathSnap results.")

PASS: Synthetic data created
PASS: Five models defined
PASS: Five-model CV completed
PASS: Out-of-fold predictions available
PASS: Prediction output schema correct
PASS: All candidate rows retained
PASS: Probabilities sum to one per observation

All framework validation checks passed.
Reminder: synthetic framework tests only, not real PathSnap results.


## 12. Optional export

Uncomment the following lines if you want CSV copies of the synthetic test outputs. Keep these clearly labelled as synthetic and separate from real experiment results.

In [12]:
# cv_results_df.to_csv("synthetic_cv_comparison.csv", index=False)
# interface_predictions.to_csv("synthetic_candidate_predictions.csv", index=False)

print("Notebook execution complete.")

Notebook execution complete.
